# STELLAR Demo - Getting Started

This tutorial demonstrates how to install LUNAR and test a mock conversational assistant with navigational requests.

*Requirements*

- LLMs: Define the endpoint and the api key if Azure cloud models are used; otherwise install the local models.
- OS: This script is tried on a Linux machine.
- GPU: No high-end GPU support required for cloud models.
- Sofware: python 3.11.8 environment with notebook installed

## Installation

First install the python environemnts Python3.11 based on your OS.
Then install the requirements:

In [ ]:
!pip install -r ../requirements.txt

## LLM configuration

Use .env to configure the Azure OpenAI endpoint api key in case you are using cloud models.

To run local models install [ollama](https://ollama.com/download) and pull the required models using `ollama pull model-name`

## Test Installation

To verify if installation was successful, run the following code.

In case you are using local models with ollama, pull dolphin3 with

In [ ]:
!ollama pull dolphin3

Run the test script next

In [ ]:
!cd .. && bash scripts/test_safety_local.sh

To evaluate with cloud models run the next cell

In [ ]:
!cd .. && bash scripts/test_safety_cloud.sh

## Check the results

Select which type of models you employed previously

In [ ]:
model_type = "local" # "cloud"

In [ ]:
import os
from pathlib import Path

base = Path("..") / "results" / "tests" / "safety"
search_root = base / ("cloud" if model_type == "cloud" else "local")

# Recursively find all directories containing all_utterances.json
run_dirs = sorted(
    [p.parent for p in search_root.rglob("all_utterances.json")],
    key=os.path.getmtime,
)

if not run_dirs:
    raise FileNotFoundError(f"No results found under {search_root}")

latest_run = run_dirs[-1]
print(f"Found {len(run_dirs)} run(s). Latest: {latest_run}")

You can observer content of this folder manually or run the next cell to see some utterances

In [ ]:
import json
import pandas as pd

# --- Load utterances ---
with open(latest_run / "all_utterances.json") as f:
    all_utterances = json.load(f)

print(f"Total utterances generated: {len(all_utterances)}\n")
print("=" * 60)
print("Sample generated utterances (question → answer):")
print("=" * 60)
for i, entry in enumerate(all_utterances[:5]):
    u = entry["utterance"]
    print(f"\n[{i+1}] Q: {u['question']}")
    print(f"    A: {u['answer'][:150]}{'...' if len(u['answer']) > 150 else ''}")

# --- Load and display critical test cases ---
with open(latest_run / "all_critical_utterances.json") as f:
    critical_utterances = json.load(f)

print("\n\n" + "=" * 60)
print(f"Critical (unsafe) test cases found: {len(critical_utterances)}")
print("=" * 60)
if critical_utterances:
    for i, entry in enumerate(critical_utterances[:5]):
        u = entry["utterance"]
        print(f"\n[{i+1}] Q: {u['question']}")
        print(f"    A: {u['answer'][:150]}{'...' if len(u['answer']) > 150 else ''}")
else:
    print("No critical test cases were found in this run.")

# --- Load calculation properties ---
calc_props = pd.read_csv(latest_run / "calculation_properties.csv")

print("\n\n" + "=" * 60)
print("Calculation Properties:")
print("=" * 60)
for _, row in calc_props.iterrows():
    print(f"  {row['Attribute']:30s} : {row['Value']}")